# Phase 3 — Sentiment vs Stock Price Correlation

## The Key Question
Does the sentiment in an earnings call transcript predict 
how the stock moves after the call?

## Methodology
1. Load sentiment scores from Phase 2
2. Pull stock prices from Yahoo Finance
   - Price 1 day before earnings
   - Price 1 day after earnings
   - Price 1 week after earnings
3. Calculate price change %
4. Correlate sentiment with price movement
5. Visualize the relationship

## Why This Matters
This is exactly what quantitative hedge funds do with 
alternative data — find signals in non-traditional data 
sources that predict market movement.

In [1]:
import pandas as pd
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# load sentiment results from Phase 2
df_sentiment = pd.read_csv('data/sentiment_results.csv')
print("Sentiment data loaded:")
print(df_sentiment[['ticker', 'label', 'positive', 'negative']])

/Users/dhaval/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Sentiment data loaded:
  ticker    label  positive  negative
0   AAPL  Neutral    0.2519    0.0387
1   MSFT  Neutral    0.1727    0.2277
2    JPM  Neutral    0.3901    0.1434
3    JNJ  Neutral    0.3478    0.0724
4   AMZN  Neutral    0.2488    0.1129
5  GOOGL  Neutral    0.2400    0.1611
6    BAC  Neutral    0.3697    0.0937
7    PFE  Neutral    0.1927    0.1923
8      V  Neutral    0.2413    0.0895
9    UNH  Neutral    0.0884    0.3344


In [4]:
# loading filings metadata to get earnings dates
df_meta = pd.read_csv('data/filings_metadata.csv')
print("Filing dates:")
print(df_meta[['ticker', 'company', 'date']])

Filing dates:
  ticker                company        date
0   AAPL              Apple Inc  2026-04-30
1   MSFT  Microsoft Corporation  2026-04-29
2    JPM         JPMorgan Chase  2026-04-14
3    JNJ      Johnson & Johnson  2026-04-14
4   AMZN                 Amazon  2026-04-29
5  GOOGL               Alphabet  2026-04-29
6    BAC        Bank of America  2026-04-15
7    PFE                 Pfizer  2026-05-05
8      V                   Visa  2026-04-28
9    UNH     UnitedHealth Group  2026-04-21


In [7]:
def get_price_change(ticker, earnings_date):
    """
    Get stock price change around earnings date
    Returns % change 1 day and 1 week after earnings
    """
    earnings_dt = pd.to_datetime(earnings_date)
    
    # get 2 weeks of data around earnings
    start = earnings_dt - timedelta(days=5)
    end = earnings_dt + timedelta(days=10)
    
    stock = yf.Ticker(ticker)
    hist = stock.history(start=start, end=end)
    
    if hist.empty:
        print(f"  No price data for {ticker}")
        return None
    
    # get price on earnings date or closest trading day
    hist.index = hist.index.tz_localize(None)
    
    # find closest date to earnings
    available_dates = hist.index.tolist()
    
    # price before earnings (day before)
    before_dates = [d for d in available_dates if d <= earnings_dt]
    after_dates = [d for d in available_dates if d > earnings_dt]
    
    if not before_dates or not after_dates:
        print(f"  Not enough dates for {ticker}")
        return None
    
    price_before = hist.loc[before_dates[-1], 'Close']
    
    # 1 day after
    price_1d = hist.loc[after_dates[0], 'Close'] if len(after_dates) >= 1 else None
    
    # 1 week after (5 trading days)
    price_1w = hist.loc[after_dates[4], 'Close'] if len(after_dates) >= 5 else None
    
    change_1d = ((price_1d - price_before) / price_before * 100) if price_1d else None
    change_1w = ((price_1w - price_before) / price_before * 100) if price_1w else None
    
    return {
        'price_before': round(price_before, 2),
        'price_1d_after': round(price_1d, 2) if price_1d else None,
        'price_1w_after': round(price_1w, 2) if price_1w else None,
        'change_1d_pct': round(change_1d, 2) if change_1d else None,
        'change_1w_pct': round(change_1w, 2) if change_1w else None
    }

# test with Apple
print("Testing Apple...")
result = get_price_change('AAPL', '2026-04-30')
print(result)

Testing Apple...
{'price_before': np.float64(271.1), 'price_1d_after': np.float64(279.88), 'price_1w_after': np.float64(287.18), 'change_1d_pct': np.float64(3.24), 'change_1w_pct': np.float64(5.93)}


In [9]:
# merging sentiment with filing dates
df_combined = df_sentiment.merge(df_meta[['ticker', 'date']], on='ticker')

# getting price changes for all companies
price_data = []

for _, row in df_combined.iterrows():
    print(f"Fetching {row['ticker']} prices...")
    prices = get_price_change(row['ticker'], row['date'])
    
    if prices:
        price_data.append({
            'ticker': row['ticker'],
            **prices
        })
    else:
        price_data.append({'ticker': row['ticker']})

df_prices = pd.DataFrame(price_data)

# merging everything together
df_final = df_combined.merge(df_prices, on='ticker')

print("\nComplete dataset:")
print(df_final[['ticker', 'label', 'positive', 'negative', 
               'change_1d_pct', 'change_1w_pct']].to_string(index=False))

Fetching AAPL prices...
Fetching MSFT prices...
Fetching JPM prices...
Fetching JNJ prices...
Fetching AMZN prices...
Fetching GOOGL prices...
Fetching BAC prices...
Fetching PFE prices...
Fetching V prices...
Fetching UNH prices...

Complete dataset:
ticker   label  positive  negative  change_1d_pct  change_1w_pct
  AAPL Neutral    0.2519    0.0387           3.24           5.93
  MSFT Neutral    0.1727    0.2277          -3.93          -2.47
   JPM Neutral    0.3901    0.1434          -1.67           0.60
   JNJ Neutral    0.3478    0.0724          -0.60          -5.81
  AMZN Neutral    0.2488    0.1129           0.77           4.54
 GOOGL Neutral    0.2400    0.1611           9.96          13.75
   BAC Neutral    0.3697    0.0937          -1.49          -2.21
   PFE Neutral    0.1927    0.1923           0.30          -0.58
     V Neutral    0.2413    0.0895           8.26           4.12
   UNH Neutral    0.0884    0.3344           2.17           6.00


In [10]:
# pulling stock prices for all 10 companies around their earnings dates
# yfinance makes this super easy - just pass ticker and date range
# learned about this in class - much easier than scraping Yahoo Finance directly

price_data = []

for _, row in df_combined.iterrows():
    print(f"Fetching {row['ticker']} prices...")
    prices = get_price_change(row['ticker'], row['date'])
    
    if prices:
        price_data.append({
            'ticker': row['ticker'],
            **prices  # unpack the dictionary into the row
        })
    else:
        # some tickers might not have data around that date
        # (holidays, trading halts etc)
        print(f"  skipping {row['ticker']} - no price data")
        price_data.append({'ticker': row['ticker']})

df_prices = pd.DataFrame(price_data)

# now merge price data with our sentiment scores
# inner join so we only keep companies where we have both
df_final = df_combined.merge(df_prices, on='ticker')

# this is the money table - sentiment vs actual stock movement
# if our hypothesis is right, positive sentiment = positive price change
print("\nSentiment vs Stock Price Movement:")
print("positive score = how bullish the earnings call was")
print("change_1d = stock price change next trading day")
print("change_1w = stock price change over next week\n")

print(df_final[['ticker', 'label', 'positive', 'negative', 
               'change_1d_pct', 'change_1w_pct']].to_string(index=False))

Fetching AAPL prices...
Fetching MSFT prices...
Fetching JPM prices...
Fetching JNJ prices...
Fetching AMZN prices...
Fetching GOOGL prices...
Fetching BAC prices...
Fetching PFE prices...
Fetching V prices...
Fetching UNH prices...

Sentiment vs Stock Price Movement:
positive score = how bullish the earnings call was
change_1d = stock price change next trading day
change_1w = stock price change over next week

ticker   label  positive  negative  change_1d_pct  change_1w_pct
  AAPL Neutral    0.2519    0.0387           3.24           5.93
  MSFT Neutral    0.1727    0.2277          -3.93          -2.47
   JPM Neutral    0.3901    0.1434          -1.67           0.60
   JNJ Neutral    0.3478    0.0724          -0.60          -5.81
  AMZN Neutral    0.2488    0.1129           0.77           4.54
 GOOGL Neutral    0.2400    0.1611           9.96          13.75
   BAC Neutral    0.3697    0.0937          -1.49          -2.21
   PFE Neutral    0.1927    0.1923           0.30          -0.58
